# Export a grouped recipe and save a challenger

This local example includes grouped categories, ordered categories, a special
level and a continuous spline. It compares controlled refits, changes a recipe
and saves both builds. Saving leaves deployment unchanged. A recipe reconstructs
declared choices; use baseline artifacts and monitoring presets for frozen
learned knots, bases or coefficients.


In [ ]:
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
from sqlalchemy import text
from superglm import Categorical, Numeric, OrderedCategorical, Spline, SuperGLM, collapse_levels
from pricing_pipeline.notebook import (
    PricingDataset, PricingModelSpec, ModelRecipe, Log, apply_transforms,
    connect, register_model, fit_model, save_model_version,
)
from pricing_pipeline.models.config import ValidationSplitConfig

MODEL_DIR = Path.cwd() / "state" / "model-recipes-demo"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPLACE_RECIPE_FILES = True  # This disposable example may be rerun.


In [ ]:
rng = np.random.default_rng(83)
n = 420
df = pd.DataFrame({
    "id": np.arange(n), "as_of": ["2026-09-01"] * n,
    "region": np.resize(["A", "B", "C"], n),
    "bonus_malus": np.resize(["0", "1", "2", "3", "4", "Unknown"], n),
    "x": rng.uniform(0, 5, n), "z": rng.normal(size=n),
    "exposure": rng.uniform(.5, 1.5, n),
})
df["claim_count"] = rng.poisson(df.exposure * np.exp(.7 + .1 * df.x))
dataset = PricingDataset(df, name="claims", source="tutorial", key="id", as_of="as_of")
FEATURES = {
    "region": Categorical(base="A", grouping=collapse_levels(df.region, groups={"BC": ["B", "C"]})),
    "bonus_malus": OrderedCategorical(
        order=["0", "1", "2", "3", "4"], specials=["Unknown"], base="0",
        basis=Spline("cr", k=3, knot_strategy="quantile"),
    ),
    "x": Spline("cr", k=3, knot_strategy="quantile"),
}
MODEL = PricingModelSpec(
    name="RECIPE_DEMO", label="Recipe demo", model_type="frequency", deployment_slot="PRODUCTION",
    target="claim_count", features=tuple(FEATURES), dataset=dataset,
    transforms={"log_exposure": Log("exposure")}, offset_column="log_exposure",
    export_weight_column="exposure", validation=ValidationSplitConfig.kfold(n_splits=3),
)
glm = SuperGLM(features=FEATURES, selection_penalty=0.0, retain_fit_state=False)


## Export before framework training

The file retains singleton A and grouped B/C, the ordered numeric positions,
Unknown as a free special level, and the spline declaration. Constructor defaults
are explicit. TOML has no null, so `{ none = true }` records an unset option.
The SQL revision number never appears in this editable file.


In [ ]:
recipe = ModelRecipe.from_model(glm, spec=MODEL)
recipe.save(MODEL_DIR / "model.toml", replace=REPLACE_RECIPE_FILES)
print((MODEL_DIR / "model.toml").read_text())


In [ ]:
loaded = ModelRecipe.load(MODEL_DIR / "model.toml")
reloaded_spec, reloaded_glm = loaded.build(dataset=dataset)
df = apply_transforms(dataset.df, MODEL.transforms)
X, y = df[list(MODEL.features)], df[MODEL.target]
prototype = glm.clone_unfitted().fit_reml(X, y, offset=df.log_exposure)
reloaded_glm.fit_reml(X, y, offset=df.log_exposure)
np.testing.assert_allclose(prototype.predict(X, offset=df.log_exposure), reloaded_glm.predict(X, offset=df.log_exposure), rtol=1e-10)
assert ModelRecipe.from_model(prototype, spec=MODEL).sha256 == loaded.sha256


## Train and save through the existing workflow

Loading does no fitting or SQL work. These calls create the fitted build and
save its version. SQL allocates the recipe revision automatically.


In [ ]:
pricing = connect(mode="local", local_root=MODEL_DIR / ".local")
MODEL, glm = loaded.build(dataset=dataset)
df = apply_transforms(dataset.df, MODEL.transforms)
model = register_model(pricing, MODEL, source_root=MODEL_DIR)
candidate = fit_model(pricing, model=model, frame=df, superglm_model=glm)
saved = save_model_version(pricing, candidate)
candidate.recipe.save(MODEL_DIR / "fitted_model.toml", replace=REPLACE_RECIPE_FILES)
display(candidate.metrics)


## Add a challenger feature

Edit a copy of the file to add one feature, then reload it against the dataset.
The feature order is explicit because it affects model construction. Existing
groups and special levels stay in the recipe. Python overrides made before
fit_model would also be recaptured.


In [ ]:
import tomllib
import tomli_w

with (MODEL_DIR / "model.toml").open("rb") as stream:
    challenger_document = tomllib.load(stream)
challenger_document["features"]["z"] = {"type": "Numeric"}
challenger_document["feature_order"].append("z")
(MODEL_DIR / "challenger.toml").write_text(tomli_w.dumps(challenger_document), encoding="utf-8")
challenger_recipe = ModelRecipe.load(MODEL_DIR / "challenger.toml")
CHALLENGER, challenger_glm = challenger_recipe.build(dataset=dataset)
challenger_model = register_model(pricing, CHALLENGER, source_root=MODEL_DIR)
challenger = fit_model(pricing, model=challenger_model, frame=df, superglm_model=challenger_glm)
challenger_saved = save_model_version(pricing, challenger)
assert challenger_saved.recipe_revision != saved.recipe_revision
with pricing.engine.connect() as connection:
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.PRICING_MODEL_DEPLOYMENT")).scalar_one() == 0
display(pd.DataFrame([
    {"Model": "Original", "Recipe": saved.recipe_revision, "Package": saved.package_version, **candidate.metrics},
    {"Model": "Challenger", "Recipe": challenger_saved.recipe_revision, "Package": challenger_saved.package_version, **challenger.metrics},
]))
pricing.engine.dispose()
